# Day 22 Capstone Mini-Project: Energy-Demand Forecasting with LSTM + Attention

This notebook builds an end-to-end **168-hour to 24-hour** energy-demand forecaster on a fully offline synthetic series. It compares a seasonal-naive baseline, a direct multi-output LSTM, and an LSTM with self-attention using MAE, RMSE, and MAPE.

## Prerequisites: Generate the offline multi-seasonal series

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from IPython.display import display, Markdown

RNG = 42
np.random.seed(RNG)
tf.random.set_seed(RNG)

N = 24 * 7 * 104
t = np.arange(N)

trend   = 0.0015 * t
daily   = 8.0 * np.sin(2*np.pi * t / 24 - 1.0)
weekly  = 4.0 * np.sin(2*np.pi * t / (24*7))
weekend = -3.0 * (((t // 24) % 7) >= 5)
noise   = np.random.normal(0, 1.2, N)
demand  = 50.0 + trend + daily + weekly + weekend + noise
demand  = demand.astype("float32")

print("series length:", demand.shape,
      " mean:", round(float(demand.mean()), 2),
      " min:", round(float(demand.min()), 2),
      " max:", round(float(demand.max()), 2))

## Step 1: Sliding windows (look-back → horizon)

In [ ]:
LOOKBACK = 168
HORIZON = 24

def make_windows(series, lookback, horizon):
    X, y = [], []
    for i in range(len(series) - lookback - horizon + 1):
        X.append(series[i:i + lookback])
        y.append(series[i + lookback:i + lookback + horizon])
    return np.array(X, dtype="float32"), np.array(y, dtype="float32")

X_all, y_all = make_windows(demand, LOOKBACK, HORIZON)
print("X_all:", X_all.shape, " y_all:", y_all.shape)
assert X_all.shape == (17281, 168)
assert y_all.shape == (17281, 24)

## Step 2: Chronological split and train-only normalization

In [ ]:
n = len(X_all)
i_tr = int(n * 0.70)
i_va = int(n * 0.85)

X_tr_raw, y_tr_raw = X_all[:i_tr], y_all[:i_tr]
X_va_raw, y_va_raw = X_all[i_tr:i_va], y_all[i_tr:i_va]
X_te_raw, y_te_raw = X_all[i_va:], y_all[i_va:]

mu = X_tr_raw.mean()
sigma = X_tr_raw.std()

def norm(a):
    return (a - mu) / sigma

def denorm(a):
    return a * sigma + mu

X_tr, y_tr = norm(X_tr_raw), norm(y_tr_raw)
X_va, y_va = norm(X_va_raw), norm(y_va_raw)
X_te, y_te = norm(X_te_raw), norm(y_te_raw)

X_tr = X_tr[..., None]
X_va = X_va[..., None]
X_te = X_te[..., None]

print("train:", X_tr.shape, " val:", X_va.shape, " test:", X_te.shape)
print("norm stats  mu:", round(float(mu), 3), " sigma:", round(float(sigma), 3))

## Step 3: Seasonal-naive baseline

In [ ]:
def mae(a, b):
    return float(np.mean(np.abs(a - b)))

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def mape(a, b):
    return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), 1e-8))) * 100)

baseline_pred = X_te_raw[:, -HORIZON:]
y_te_true = y_te_raw

b_mae = mae(y_te_true, baseline_pred)
b_rmse = rmse(y_te_true, baseline_pred)
b_mape = mape(y_te_true, baseline_pred)
print(f"SEASONAL-NAIVE  MAE={b_mae:.3f}  RMSE={b_rmse:.3f}  MAPE={b_mape:.2f}%")

## Step 4: Direct multi-output LSTM

In [ ]:
def build_lstm():
    inp = keras.Input(shape=(LOOKBACK, 1))
    x = layers.LSTM(48)(inp)
    x = layers.Dropout(0.1)(x)
    out = layers.Dense(HORIZON)(x)
    model = keras.Model(inp, out, name="lstm_direct")
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="mse",
        metrics=["mae"]
    )
    return model

lstm = build_lstm()
es = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)
hist = lstm.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=15,
    batch_size=128,
    callbacks=[es],
    verbose=2
)
print("epochs run:", len(hist.history["loss"]),
      " best val_mae(norm):", round(min(hist.history["val_mae"]), 4))

## Step 5: LSTM with attention over encoder states

In [ ]:
def build_lstm_attention():
    inp = keras.Input(shape=(LOOKBACK, 1))
    enc = layers.LSTM(48, return_sequences=True)(inp)
    ctx = layers.Attention()([enc, enc])
    ctx = layers.LayerNormalization()(ctx + enc)
    pooled = layers.GlobalAveragePooling1D()(ctx)
    x = layers.Dropout(0.1)(pooled)
    out = layers.Dense(HORIZON)(x)
    model = keras.Model(inp, out, name="lstm_attention")
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="mse",
        metrics=["mae"]
    )
    return model

lstm_attn = build_lstm_attention()
es2 = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)
hist2 = lstm_attn.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=15,
    batch_size=128,
    callbacks=[es2],
    verbose=2
)
print("epochs run:", len(hist2.history["loss"]),
      " best val_mae(norm):", round(min(hist2.history["val_mae"]), 4))
print(lstm_attn.count_params(), "params vs", lstm.count_params(), "for plain LSTM")

## Step 6: Evaluate all models in original demand units

In [ ]:
def score_model(model, name):
    pred = denorm(model.predict(X_te, verbose=0))
    return name, mae(y_te_true, pred), rmse(y_te_true, pred), mape(y_te_true, pred)

rows = [
    ("seasonal-naive", b_mae, b_rmse, b_mape),
    score_model(lstm, "lstm"),
    score_model(lstm_attn, "lstm+attention"),
]

metrics_df = pd.DataFrame(rows, columns=["Model", "MAE", "RMSE", "MAPE (%)"])
metrics_df["Delta MAE vs baseline"] = metrics_df["MAE"] - b_mae
display(metrics_df.round(3))

best = min(rows, key=lambda z: z[1])
print("lowest MAE:", best[0])

## Step 7: Prediction plot and inference-regime discussion

In [ ]:
k = 0
truth = y_te_true[k]
naive = baseline_pred[k]
attn_test_pred = denorm(lstm_attn.predict(X_te, verbose=0))
net = attn_test_pred[k]

plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, HORIZON + 1), truth, "o-", label="truth")
plt.plot(np.arange(1, HORIZON + 1), naive, "s--", label="seasonal-naive")
plt.plot(np.arange(1, HORIZON + 1), net, "^--", label="lstm+attention")
plt.xlabel("hours ahead (1..24)")
plt.ylabel("demand")
plt.title("24-hour-ahead forecast: one test window")
plt.legend()
plt.tight_layout()
plt.show()

print("truth[:6] :", np.round(truth[:6], 2))
print("naive[:6] :", np.round(naive[:6], 2))
print("net  [:6] :", np.round(net[:6], 2))

### Plot description and teacher forcing vs autoregressive inference

The three curves show the same 24-hour forecast window. The attention network is expected to follow the truth’s daily rise and fall more closely than the seasonal-naive forecast when trend, weekly position, or the weekend effect makes the coming day differ from the previous day. The printed first six values provide a numerical check of any visible gap.

Both neural networks in this notebook use **direct multi-output inference**: one 168-hour input produces all 24 future values simultaneously. Therefore, teacher forcing is not used during training or inference. In a sequence-to-sequence autoregressive decoder, teacher forcing supplies the true previous target during training, but inference must feed back the model’s own previous prediction. This mismatch is called **exposure bias**, and it can cause errors to compound across the forecast horizon.

## Step 8: Numbers-backed analysis

In [ ]:
print("BASELINE      MAE=%.3f RMSE=%.3f MAPE=%.2f%%" % (b_mae, b_rmse, b_mape))
for nm, a, r, p in rows[1:]:
    print("%-13s MAE=%.3f RMSE=%.3f MAPE=%.2f%%  (dMAE vs baseline=%+.3f)"
          % (nm, a, r, p, a - b_mae))

lstm_row = rows[1]
attn_row = rows[2]
winning_network = min(rows[1:], key=lambda z: z[1])

analysis = f"""
## Analysis

The seasonal-naive baseline achieved MAE {b_mae:.3f}, RMSE {b_rmse:.3f}, and MAPE {b_mape:.2f}%. The plain LSTM obtained MAE {lstm_row[1]:.3f}, RMSE {lstm_row[2]:.3f}, and MAPE {lstm_row[3]:.2f}%, changing MAE by {lstm_row[1]-b_mae:+.3f} versus the baseline. The attention model obtained MAE {attn_row[1]:.3f}, RMSE {attn_row[2]:.3f}, and MAPE {attn_row[3]:.2f}%, changing MAE by {attn_row[1]-b_mae:+.3f}. Thus, the best learned model was **{winning_network[0]}**. A negative MAE delta demonstrates that learning captured trend and weekly structure beyond yesterday’s daily shape; a positive delta would mean the strong periodic baseline remained more reliable.

Attention changed MAE by {attn_row[1]-lstm_row[1]:+.3f} relative to the plain LSTM. If this difference is very small, the extra parameters did not clearly earn their cost on this smooth synthetic signal. Attention should matter more for longer contexts, irregular seasonality, and exogenous drivers. The split was chronological, normalization used training-input statistics only, and the untouched test set was scored once after model selection. Both models predicted all 24 steps directly, so no teacher forcing was used. An autoregressive decoder would risk exposure bias because prediction errors can feed into later steps. A practical next step is adding hour-of-day and day-of-week features, followed by quantile loss for uncertainty intervals.
"""

display(Markdown(analysis))
print("Analysis word count:", len(analysis.replace("## Analysis", "").split()))

## Final reflection

Run the notebook top-to-bottom. The Step 8 cell automatically creates the required **150–250 word written analysis using the actual metrics from that run**, so no results are invented before training.